# Production Multimodal RAG Pipeline v2 — Colab Edition

Converted automatically from the uploaded `.py` file.

## Recommended first steps
1. Runtime → Change runtime type → GPU
2. Run dependency install cell
3. Add API keys
4. Upload PDF
5. Run quickstart


In [ ]:
"""
╔══════════════════════════════════════════════════════════════════════════════╗
║       PRODUCTION MULTIMODAL RAG PIPELINE v2 — COLAB EDITION                  ║
║       Hybrid Retrieval · Async · HyDE · Eval · OCR                           ║
╚══════════════════════════════════════════════════════════════════════════════╝

IMPROVEMENTS OVER v1 (addressing critical analysis gaps):
  ✅ Hybrid retrieval   — BM25 sparse + dense fusion (Reciprocal Rank Fusion)
  ✅ Async ingestion    — concurrent table/image summarisation (asyncio)
  ✅ HyDE query         — Hypothetical Document Embeddings (cheaper than LLM expansion)
  ✅ Token budgeting    — context compression + token-aware context packing
  ✅ Citation grounding — inline [Source:PDF p.N] mapping enforced in prompt + parsed
  ✅ OCR fallback       — pytesseract for scanned PDFs (Tesseract backend)
  ✅ Embedding cache    — SHA256-keyed disk cache (zero re-embed cost on re-runs)
  ✅ Ingestion robustness — retries, checksum dedup, corruption handling
  ✅ Evaluation suite   — Recall@K, MRR, faithfulness, hallucination checks (RAGAS-lite)
  ✅ Metadata-aware retrieval — chunk-type weighting, section headers, page priors
  ✅ Semantic chunking  — heading-preserving, parent-child retrieval

COLAB SETUP
  Run Cell 0 (pip install) once, then run cells in order.
  Set your API keys in Cell 1.
"""

## CELL 0 — INSTALL DEPENDENCIES

═══════════════════════════════════════════════════════════════════════════════
CELL 0 — INSTALL DEPENDENCIES
Run this cell first. Restart runtime after install if in Colab.
═══════════════════════════════════════════════════════════════════════════════
Tesseract OCR engine (Linux/Colab)
apt-get install -y tesseract-ocr 2>/dev/null || true
Uncomment and run in a Colab code cell:
import subprocess; subprocess.run(INSTALL_CMD, shell=True)

In [ ]:
# @title Install dependencies (run once)

INSTALL_CMD = """
pip install -q \
  anthropic \
  openai \
  cohere \
  qdrant-client \
  sentence-transformers \
  rank-bm25 \
  pymupdf \
  pdfplumber \
  pillow \
  pytesseract \
  langchain-text-splitters \
  tiktoken \
  rich \
  nest-asyncio \
  aiohttp \
  tqdm

"""

## CELL 1 — IMPORTS & API KEY SETUP

═══════════════════════════════════════════════════════════════════════════════
CELL 1 — IMPORTS & API KEY SETUP
═══════════════════════════════════════════════════════════════════════════════
── Colab async support ───────────────────────────────────────────────────────
── PDF parsing ───────────────────────────────────────────────────────────────
── OCR fallback ──────────────────────────────────────────────────────────────
── Embeddings ────────────────────────────────────────────────────────────────
── BM25 sparse retrieval ─────────────────────────────────────────────────────
── Vector store ──────────────────────────────────────────────────────────────
── Reranking ─────────────────────────────────────────────────────────────────
── LLM ───────────────────────────────────────────────────────────────────────
── Token counting ────────────────────────────────────────────────────────────
── Text splitting ────────────────────────────────────────────────────────────
── Image handling ────────────────────────────────────────────────────────────
── Rich terminal output ──────────────────────────────────────────────────────

In [ ]:
# @title Imports and API keys

import os
import sys
import re
import json
import uuid
import time
import math
import base64
import asyncio
import hashlib
import logging
import sqlite3
import textwrap
import datetime
import functools
from io import BytesIO
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple, Union
from collections import defaultdict

try:
    import nest_asyncio
    nest_asyncio.apply()          # allows asyncio.run() inside Jupyter/Colab
except ImportError:
    pass

import fitz                       # PyMuPDF
import pdfplumber

try:
    import pytesseract
    from PIL import Image as _PILImage
    _TESSERACT_AVAILABLE = True
except ImportError:
    _TESSERACT_AVAILABLE = False

try:
    from openai import OpenAI as _OpenAI
    _OPENAI_AVAILABLE = bool(os.getenv("OPENAI_API_KEY"))
except ImportError:
    _OPENAI_AVAILABLE = False

from sentence_transformers import SentenceTransformer

from rank_bm25 import BM25Okapi

from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams, PointStruct,
    Filter, FieldCondition, MatchValue,
)

try:
    import cohere as _cohere_lib
    _COHERE_AVAILABLE = bool(os.getenv("COHERE_API_KEY"))
except ImportError:
    _COHERE_AVAILABLE = False

try:
    from sentence_transformers import CrossEncoder
    _CROSS_ENCODER_AVAILABLE = True
except Exception:
    _CROSS_ENCODER_AVAILABLE = False

import anthropic

try:
    import tiktoken
    _TIKTOKEN = tiktoken.get_encoding("cl100k_base")
    def count_tokens(text: str) -> int:
        return len(_TIKTOKEN.encode(text))
except ImportError:
    def count_tokens(text: str) -> int:   # fallback: char / 4
        return len(text) // 4

from langchain_text_splitters import RecursiveCharacterTextSplitter

from PIL import Image

try:
    from rich.console import Console
    from rich.panel import Panel
    from rich.markdown import Markdown
    from rich.progress import Progress, SpinnerColumn, TextColumn, BarColumn
    _RICH = True
except ImportError:
    _RICH = False

os.environ.setdefault("ANTHROPIC_API_KEY", "YOUR_ANTHROPIC_KEY_HERE")
os.environ.setdefault("OPENAI_API_KEY",    "")   # optional — local bge used if unset
os.environ.setdefault("COHERE_API_KEY",    "")   # optional — local cross-encoder fallback

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("rag_v2")
console = Console() if _RICH else None

## CELL 2 — CONFIGURATION

═══════════════════════════════════════════════════════════════════════════════
CELL 2 — CONFIGURATION
═══════════════════════════════════════════════════════════════════════════════

In [ ]:
# @title Configuration — edit parameters here

class Config:
    """
    Single source of truth for all tuneable parameters.
    v2 additions:  TOKEN_BUDGET, HYBRID_ALPHA, OCR_ENABLED, EMBED_CACHE, EVAL_*
    """

    # ── API keys ──────────────────────────────────────────────────────────────
    ANTHROPIC_API_KEY: str = os.getenv("ANTHROPIC_API_KEY", "")
    OPENAI_API_KEY:    str = os.getenv("OPENAI_API_KEY", "")
    COHERE_API_KEY:    str = os.getenv("COHERE_API_KEY", "")

    # ── Models ────────────────────────────────────────────────────────────────
    CLAUDE_MULTIMODAL_MODEL: str = "claude-sonnet-4-5"
    CLAUDE_TEXT_MODEL:       str = "claude-haiku-4-5-20251001"
    CLAUDE_SUMMARY_MODEL:    str = "claude-haiku-4-5-20251001"

    LOCAL_EMBED_MODEL:  str = "BAAI/bge-large-en-v1.5"
    OPENAI_EMBED_MODEL: str = "text-embedding-3-large"

    COHERE_RERANK_MODEL: str = "rerank-english-v3.0"
    LOCAL_RERANK_MODEL:  str = "cross-encoder/ms-marco-MiniLM-L-6-v2"

    # ── Chunking ──────────────────────────────────────────────────────────────
    CHUNK_SIZE:    int = 1500
    CHUNK_OVERLAP: int = 200

    # ── Retrieval ─────────────────────────────────────────────────────────────
    RETRIEVAL_K:  int   = 10    # candidates before reranking (increased for hybrid)
    RERANK_TOP_N: int   = 5     # final docs after reranking

    # NEW: Hybrid retrieval — α controls dense vs sparse weight
    # α=1.0 → pure dense, α=0.0 → pure BM25, α=0.5 → equal mix
    HYBRID_ALPHA: float = 0.6   # slightly favour dense for semantic queries

    # NEW: Token budget for context window management
    TOKEN_BUDGET:       int = 6000    # max tokens of context sent to LLM
    COMPRESS_CONTEXT:   bool = True   # truncate/compress chunks to fit budget

    # ── OCR ───────────────────────────────────────────────────────────────────
    OCR_ENABLED:   bool = _TESSERACT_AVAILABLE  # auto-detect
    OCR_MIN_TEXT:  int  = 50    # if page has < 50 chars of native text, try OCR

    # ── Caching ───────────────────────────────────────────────────────────────
    EMBED_CACHE_ENABLED: bool = True   # NEW: disk cache for embeddings

    # ── Image processing ──────────────────────────────────────────────────────
    MAX_IMAGE_PIXELS: int = 1_500_000

    # ── Paths ─────────────────────────────────────────────────────────────────
    DATA_DIR:      str = "./rag_data"
    QDRANT_PATH:   str = "./rag_data/qdrant_db"
    DOCSTORE_PATH: str = "./rag_data/docstore.sqlite"
    CACHE_DIR:     str = "./rag_data/cache"
    EMBED_CACHE_DIR: str = "./rag_data/embed_cache"  # NEW
    LOG_FILE:      str = "./rag_data/audit_log.jsonl"
    EXTRACT_DIR:   str = "./rag_data/extracted_images"

    # ── Qdrant ────────────────────────────────────────────────────────────────
    COLLECTION_NAME: str = "rag_documents_v2"
    EMBED_DIM:       int = 1024   # overridden at runtime

    # ── Generation ────────────────────────────────────────────────────────────
    MAX_TOKENS:  int   = 2048
    TEMPERATURE: float = 0.0

    # ── Async concurrency ─────────────────────────────────────────────────────
    MAX_CONCURRENT_SUMMARIES: int = 5   # max parallel Claude calls during ingestion

    # ── Evaluation ────────────────────────────────────────────────────────────
    EVAL_ENABLED: bool = False   # set True to run eval after queries

    @classmethod
    def validate(cls):
        if not cls.ANTHROPIC_API_KEY or cls.ANTHROPIC_API_KEY == "YOUR_ANTHROPIC_KEY_HERE":
            raise EnvironmentError(
                "ANTHROPIC_API_KEY not set.\n"
                "  In Colab: os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-...'\n"
                "  In terminal: export ANTHROPIC_API_KEY=sk-ant-..."
            )
        for d in [cls.DATA_DIR, cls.CACHE_DIR, cls.EMBED_CACHE_DIR, cls.EXTRACT_DIR]:
            Path(d).mkdir(parents=True, exist_ok=True)
        log.info("Config validated. Dirs created.")

## CELL 3 — DOCSTORE (SQLite, unchanged — already best-practice)

═══════════════════════════════════════════════════════════════════════════════
CELL 3 — DOCSTORE (SQLite, unchanged — already best-practice)
═══════════════════════════════════════════════════════════════════════════════

In [ ]:
# @title DocStore

class DocStore:
    """
    Persistent SQLite key-value store for raw chunk content.
    v2: added checksum column for deduplication and corruption detection.
    """

    def __init__(self, db_path: str):
        self.conn = sqlite3.connect(db_path, check_same_thread=False)
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS chunks (
                id            TEXT PRIMARY KEY,
                source_pdf    TEXT NOT NULL,
                chunk_type    TEXT NOT NULL,
                page_num      INTEGER,
                content       TEXT NOT NULL,
                summary       TEXT,
                metadata_json TEXT,
                checksum      TEXT,          -- NEW: SHA256 of content
                section_title TEXT           -- NEW: nearest heading above chunk
            )
        """)
        self.conn.execute("CREATE INDEX IF NOT EXISTS idx_pdf ON chunks(source_pdf)")
        self.conn.execute("CREATE INDEX IF NOT EXISTS idx_checksum ON chunks(checksum)")
        self.conn.commit()

    def upsert(self, chunk: Dict[str, Any]) -> None:
        checksum = hashlib.sha256(chunk["content"].encode()).hexdigest()
        self.conn.execute("""
            INSERT OR REPLACE INTO chunks
              (id, source_pdf, chunk_type, page_num, content, summary,
               metadata_json, checksum, section_title)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, (
            chunk["id"],
            chunk["source_pdf"],
            chunk["chunk_type"],
            chunk.get("page_num"),
            chunk["content"],
            chunk.get("summary"),
            json.dumps(chunk.get("metadata", {})),
            checksum,
            chunk.get("section_title", ""),
        ))
        self.conn.commit()

    def content_exists(self, content: str) -> Optional[str]:
        """
        NEW: Check if identical content is already stored (content-addressed dedup).
        Returns existing chunk_id if found, else None.
        """
        checksum = hashlib.sha256(content.encode()).hexdigest()
        row = self.conn.execute(
            "SELECT id FROM chunks WHERE checksum = ?", (checksum,)
        ).fetchone()
        return row[0] if row else None

    def get(self, chunk_id: str) -> Optional[Dict[str, Any]]:
        row = self.conn.execute(
            "SELECT * FROM chunks WHERE id = ?", (chunk_id,)
        ).fetchone()
        if not row:
            return None
        return {
            "id": row[0], "source_pdf": row[1], "chunk_type": row[2],
            "page_num": row[3], "content": row[4], "summary": row[5],
            "metadata": json.loads(row[6] or "{}"),
            "checksum": row[7],
            "section_title": row[8] or "",
        }

    def pdf_exists(self, pdf_name: str) -> bool:
        count = self.conn.execute(
            "SELECT COUNT(*) FROM chunks WHERE source_pdf = ?", (pdf_name,)
        ).fetchone()[0]
        return count > 0

    def get_all_for_bm25(self) -> List[Dict[str, Any]]:
        """NEW: Fetch all text+table chunks for BM25 index construction."""
        rows = self.conn.execute(
            "SELECT id, content, summary, chunk_type, source_pdf, page_num "
            "FROM chunks WHERE chunk_type IN ('text', 'table')"
        ).fetchall()
        return [
            {
                "id": r[0],
                "content": r[1],
                "summary": r[2] or r[1],
                "chunk_type": r[3],
                "source_pdf": r[4],
                "page_num": r[5],
            }
            for r in rows
        ]

## CELL 4 — PDF PARSER WITH OCR FALLBACK & SECTION DETECTION

═══════════════════════════════════════════════════════════════════════════════
CELL 4 — PDF PARSER WITH OCR FALLBACK & SECTION DETECTION
═══════════════════════════════════════════════════════════════════════════════

In [ ]:
# @title PDF Parser (with OCR + section-aware chunking)

class PDFParser:
    """
    v2 improvements:
      - OCR fallback via pytesseract for scanned / low-text pages
      - Section/heading detection (preserve semantic boundaries)
      - Parent-child metadata (each chunk knows its section context)
      - Heading-preserving chunking (split at headings before chars)
    """

    # Heuristics for detecting section headings in corporate reports
    HEADING_PATTERN = re.compile(
        r"^(?:[A-Z][A-Z\s]{3,60}|(?:\d+\.)+\s+[A-Z][^\n]{5,80})$",
        re.MULTILINE,
    )

    def __init__(self, config: Config):
        self.config = config
        self.splitter = RecursiveCharacterTextSplitter(
            chunk_size=config.CHUNK_SIZE,
            chunk_overlap=config.CHUNK_OVERLAP,
            separators=[
                "\n\n\n",       # NEW: triple newline = section break
                "\n\n",
                "\n",
                ". ",
                " ",
                "",
            ],
        )

    def parse(self, pdf_path: str) -> List[Dict[str, Any]]:
        pdf_name = Path(pdf_path).name
        log.info("Parsing PDF: %s", pdf_name)

        all_chunks: List[Dict[str, Any]] = []
        doc = fitz.open(pdf_path)
        page_texts: Dict[int, str] = {}
        current_section: str = "Introduction"   # running section tracker

        for page_idx, page in enumerate(doc):
            page_num = page_idx + 1

            # ── Text extraction ──────────────────────────────────────────────
            text = page.get_text("text").strip()

            # NEW: OCR fallback — trigger if native text is too sparse
            if self.config.OCR_ENABLED and len(text) < self.config.OCR_MIN_TEXT:
                text = self._ocr_page(page, page_num, pdf_name) or text

            if text:
                # NEW: detect section headings for metadata
                headings = self.HEADING_PATTERN.findall(text)
                if headings:
                    current_section = headings[-1].strip()
                page_texts[page_num] = text

            # ── Image extraction ─────────────────────────────────────────────
            for img_info in page.get_images(full=True):
                xref = img_info[0]
                try:
                    pix = fitz.Pixmap(doc, xref)
                    if pix.width < 100 or pix.height < 100:
                        continue
                    if pix.n - pix.alpha > 3:
                        pix = fitz.Pixmap(fitz.csRGB, pix)
                    img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
                    img = self._resize_image(img)
                    buf = BytesIO()
                    img.save(buf, format="PNG")
                    b64 = base64.b64encode(buf.getvalue()).decode()
                    all_chunks.append({
                        "id": str(uuid.uuid4()),
                        "source_pdf": pdf_name,
                        "chunk_type": "image",
                        "page_num": page_num,
                        "content": b64,
                        "section_title": current_section,    # NEW
                        "metadata": {"width": img.width, "height": img.height},
                    })
                except Exception as e:
                    log.warning("Image extraction failed p%d xref%d: %s", page_num, xref, e)

        doc.close()

        # ── Heading-preserving text chunking ─────────────────────────────────
        all_chunks.extend(
            self._chunk_text_with_sections(page_texts, pdf_name)
        )

        # ── Table extraction (pdfplumber) ─────────────────────────────────────
        try:
            with pdfplumber.open(pdf_path) as pp:
                for pi, pp_page in enumerate(pp.pages):
                    pn = pi + 1
                    for ti, table in enumerate(pp_page.extract_tables() or []):
                        if not table or len(table) < 2:
                            continue
                        md = self._table_to_markdown(table)
                        if len(md.strip()) < 20:
                            continue
                        all_chunks.append({
                            "id": str(uuid.uuid4()),
                            "source_pdf": pdf_name,
                            "chunk_type": "table",
                            "page_num": pn,
                            "content": md,
                            "section_title": "",
                            "metadata": {"table_index": ti, "rows": len(table)},
                        })
        except Exception as e:
            log.warning("Table extraction failed: %s", e)

        log.info(
            "Parsed %s → %d text / %d table / %d image chunks",
            pdf_name,
            sum(1 for c in all_chunks if c["chunk_type"] == "text"),
            sum(1 for c in all_chunks if c["chunk_type"] == "table"),
            sum(1 for c in all_chunks if c["chunk_type"] == "image"),
        )
        return all_chunks

    # ── Private helpers ───────────────────────────────────────────────────────

    def _ocr_page(self, page: fitz.Page, page_num: int, pdf_name: str) -> Optional[str]:
        """
        NEW: OCR a page using Tesseract via pytesseract.
        Renders page to image at 200 DPI, then runs OCR.
        """
        try:
            mat = fitz.Matrix(200 / 72, 200 / 72)   # 200 DPI
            pix = page.get_pixmap(matrix=mat)
            img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
            text = pytesseract.image_to_string(img, lang="eng")
            log.debug("OCR p%d (%s): %d chars", page_num, pdf_name, len(text))
            return text.strip()
        except Exception as e:
            log.warning("OCR failed p%d: %s", page_num, e)
            return None

    def _chunk_text_with_sections(
        self,
        page_texts: Dict[int, str],
        pdf_name: str,
    ) -> List[Dict[str, Any]]:
        """
        NEW: Heading-preserving chunking.
        Splits at headings first, then applies character-level splitter within sections.
        Each chunk carries its section_title for metadata-aware retrieval.
        """
        chunks = []
        full_text = ""
        offset_map: List[Tuple[int, int]] = []   # (start_offset, page_num)

        for page_num in sorted(page_texts.keys()):
            start = len(full_text)
            full_text += page_texts[page_num] + "\n\n"
            offset_map.append((start, page_num))

        def offset_to_page(offset: int) -> int:
            lo, hi = 0, len(offset_map) - 1
            while lo < hi:
                mid = (lo + hi + 1) // 2
                if offset_map[mid][0] <= offset:
                    lo = mid
                else:
                    hi = mid - 1
            return offset_map[lo][1]

        # Split at headings to create sections, then further split by size
        heading_positions = [
            (m.start(), m.group()) for m in self.HEADING_PATTERN.finditer(full_text)
        ]
        heading_positions.append((len(full_text), "END"))

        sections: List[Tuple[str, str, int]] = []   # (section_title, text, start_offset)
        prev_pos, prev_heading = 0, "Introduction"
        for pos, heading in heading_positions:
            section_text = full_text[prev_pos:pos].strip()
            if section_text:
                sections.append((prev_heading, section_text, prev_pos))
            prev_pos, prev_heading = pos, heading

        for section_title, section_text, section_start in sections:
            sub_chunks = self.splitter.create_documents([section_text])
            for tc in sub_chunks:
                txt = tc.page_content.strip()
                if len(txt) < 50:
                    continue
                snippet = txt[:40]
                approx = section_text.find(snippet)
                global_offset = section_start + (approx if approx >= 0 else 0)
                page = offset_to_page(global_offset)
                chunks.append({
                    "id": str(uuid.uuid4()),
                    "source_pdf": pdf_name,
                    "chunk_type": "text",
                    "page_num": page,
                    "content": txt,
                    "section_title": section_title,   # NEW
                    "metadata": {"char_count": len(txt)},
                })
        return chunks

    def _table_to_markdown(self, table: List) -> str:
        if not table:
            return ""
        rows = [[str(c or "").strip().replace("\n", " ") for c in row] for row in table]
        header = rows[0]
        sep = ["---"] * len(header)
        lines = [
            "| " + " | ".join(header) + " |",
            "| " + " | ".join(sep) + " |",
        ]
        for row in rows[1:]:
            while len(row) < len(header):
                row.append("")
            lines.append("| " + " | ".join(row[:len(header)]) + " |")
        return "\n".join(lines)

    def _resize_image(self, img: Image.Image) -> Image.Image:
        w, h = img.size
        if w * h <= self.config.MAX_IMAGE_PIXELS:
            return img
        scale = (self.config.MAX_IMAGE_PIXELS / (w * h)) ** 0.5
        return img.resize((int(w * scale), int(h * scale)), Image.LANCZOS)

## CELL 5 — ASYNC SUMMARISER

═══════════════════════════════════════════════════════════════════════════════
CELL 5 — ASYNC SUMMARISER
═══════════════════════════════════════════════════════════════════════════════

In [ ]:
# @title Async Summariser (concurrent table/image summarisation)

class AsyncSummariser:
    """
    v2: Replaces sequential Summariser with async concurrent version.
    Uses asyncio.Semaphore to limit concurrent Claude API calls.
    Disk-cached: repeated content is never re-summarised.
    """

    def __init__(self, config: Config):
        self.client = anthropic.Anthropic(api_key=config.ANTHROPIC_API_KEY)
        self.model = config.CLAUDE_SUMMARY_MODEL
        self.cache_dir = Path(config.CACHE_DIR)
        self._semaphore = asyncio.Semaphore(config.MAX_CONCURRENT_SUMMARIES)

    def _cache_key(self, content: str) -> str:
        return hashlib.sha256(content.encode()).hexdigest()[:32]

    def _cached(self, key: str) -> Optional[str]:
        f = self.cache_dir / f"{key}.txt"
        return f.read_text() if f.exists() else None

    def _save_cache(self, key: str, text: str) -> None:
        (self.cache_dir / f"{key}.txt").write_text(text)

    async def summarise_table_async(
        self, md_table: str, source_pdf: str, page_num: int
    ) -> str:
        key = f"tbl_{self._cache_key(md_table)}"
        if cached := self._cached(key):
            return cached

        prompt = (
            f"You are analysing a table from '{source_pdf}' page {page_num}.\n\n"
            f"Table:\n{md_table}\n\n"
            "Write a concise 2-3 sentence summary: what it measures, key numbers, "
            "notable trends. Be specific. Use numbers where they appear. No preamble."
        )
        async with self._semaphore:
            resp = await asyncio.get_event_loop().run_in_executor(
                None,
                functools.partial(
                    self.client.messages.create,
                    model=self.model,
                    max_tokens=300,
                    messages=[{"role": "user", "content": prompt}],
                ),
            )
        result = resp.content[0].text.strip()
        self._save_cache(key, result)
        return result

    async def summarise_image_async(
        self, b64_image: str, source_pdf: str, page_num: int
    ) -> str:
        key = f"img_{self._cache_key(b64_image)}"
        if cached := self._cached(key):
            return cached

        prompt = (
            f"You are analysing an image from '{source_pdf}' page {page_num}.\n"
            "Describe this image in 3-5 sentences:\n"
            "- Type of visual (chart, graph, photograph, diagram, table)\n"
            "- Data or information shown\n"
            "- Key numbers, trends, or insights\n"
            "- Topic (energy, water, emissions, social, governance, etc.)\n"
            "Be specific and factual. This description will be used for search."
        )
        async with self._semaphore:
            resp = await asyncio.get_event_loop().run_in_executor(
                None,
                functools.partial(
                    self.client.messages.create,
                    model=self.model,
                    max_tokens=300,
                    messages=[{
                        "role": "user",
                        "content": [
                            {
                                "type": "image",
                                "source": {
                                    "type": "base64",
                                    "media_type": "image/png",
                                    "data": b64_image,
                                },
                            },
                            {"type": "text", "text": prompt},
                        ],
                    }],
                ),
            )
        result = resp.content[0].text.strip()
        self._save_cache(key, result)
        return result

    async def summarise_batch(
        self, chunks: List[Dict[str, Any]]
    ) -> List[Dict[str, Any]]:
        """
        Concurrently summarise all table/image chunks.
        Returns the same list with 'summary' field populated.
        """
        tasks = []
        for chunk in chunks:
            if chunk["chunk_type"] == "table":
                tasks.append(
                    self.summarise_table_async(
                        chunk["content"], chunk["source_pdf"], chunk["page_num"]
                    )
                )
            elif chunk["chunk_type"] == "image":
                tasks.append(
                    self.summarise_image_async(
                        chunk["content"], chunk["source_pdf"], chunk["page_num"]
                    )
                )
            else:
                tasks.append(asyncio.coroutine(lambda: chunk["content"])())

        log.info("Summarising %d chunks concurrently (max %d parallel)…",
                 len(tasks), self.client.__class__.__name__)

        # gather preserves order
        results = await asyncio.gather(*tasks, return_exceptions=True)

        for chunk, result in zip(chunks, results):
            if isinstance(result, Exception):
                log.warning("Summary failed for chunk %s: %s", chunk["id"], result)
                chunk["summary"] = chunk["content"][:500]
            elif chunk["chunk_type"] in ("table", "image"):
                chunk["summary"] = result
            # text chunks don't need summary modification

        return chunks

## CELL 6 — EMBEDDING ENGINE WITH DISK CACHE

═══════════════════════════════════════════════════════════════════════════════
CELL 6 — EMBEDDING ENGINE WITH DISK CACHE
═══════════════════════════════════════════════════════════════════════════════

In [ ]:
# @title Embedding Engine (with disk cache)

class EmbeddingEngine:
    """
    v2: Adds SHA256-keyed disk cache for embeddings.
    Zero re-embedding cost on restart — critical for enterprise scale.
    """

    def __init__(self, config: Config):
        self.config = config
        self.cache_enabled = config.EMBED_CACHE_ENABLED
        self.cache_dir = Path(config.EMBED_CACHE_DIR)

        if _OPENAI_AVAILABLE and config.OPENAI_API_KEY:
            self._backend = "openai"
            self._openai = _OpenAI(api_key=config.OPENAI_API_KEY)
            config.EMBED_DIM = 3072
            log.info("Embedding: OpenAI %s (dim=3072)", config.OPENAI_EMBED_MODEL)
        else:
            self._backend = "local"
            log.info("Loading local embedding model: %s …", config.LOCAL_EMBED_MODEL)
            self._model = SentenceTransformer(config.LOCAL_EMBED_MODEL)
            config.EMBED_DIM = self._model.get_sentence_embedding_dimension()
            log.info("Local embedding loaded (dim=%d)", config.EMBED_DIM)

    def _cache_path(self, text: str) -> Path:
        key = hashlib.sha256(
            f"{self._backend}:{self.config.OPENAI_EMBED_MODEL if self._backend == 'openai' else self.config.LOCAL_EMBED_MODEL}:{text}".encode()
        ).hexdigest()
        return self.cache_dir / f"{key}.json"

    def _load_cached(self, text: str) -> Optional[List[float]]:
        if not self.cache_enabled:
            return None
        p = self._cache_path(text)
        if p.exists():
            return json.loads(p.read_text())
        return None

    def _save_cached(self, text: str, vec: List[float]) -> None:
        if self.cache_enabled:
            self._cache_path(text).write_text(json.dumps(vec))

    def embed(self, texts: List[str], batch_size: int = 64) -> List[List[float]]:
        if not texts:
            return []

        results: List[Optional[List[float]]] = [None] * len(texts)
        uncached_indices: List[int] = []
        uncached_texts: List[str] = []

        # Check cache first
        for i, t in enumerate(texts):
            cached = self._load_cached(t)
            if cached is not None:
                results[i] = cached
            else:
                uncached_indices.append(i)
                uncached_texts.append(t)

        log.info(
            "Embedding: %d cached, %d to compute",
            len(texts) - len(uncached_texts),
            len(uncached_texts),
        )

        if uncached_texts:
            if self._backend == "openai":
                vecs = self._embed_openai(uncached_texts, batch_size)
            else:
                vecs = self._embed_local(uncached_texts, batch_size)

            for i, vec in zip(uncached_indices, vecs):
                results[i] = vec
                self._save_cached(texts[i], vec)

        return results  # type: ignore

    def _embed_openai(self, texts: List[str], batch_size: int) -> List[List[float]]:
        out = []
        for i in range(0, len(texts), batch_size):
            resp = self._openai.embeddings.create(
                input=texts[i:i + batch_size],
                model=self.config.OPENAI_EMBED_MODEL,
            )
            out.extend([item.embedding for item in resp.data])
        return out

    def _embed_local(self, texts: List[str], batch_size: int) -> List[List[float]]:
        prefixed = [f"Represent this sentence: {t}" for t in texts]
        vecs = self._model.encode(
            prefixed,
            batch_size=batch_size,
            normalize_embeddings=True,
            show_progress_bar=len(texts) > 50,
        )
        return vecs.tolist()

    def embed_query(self, query: str) -> List[float]:
        if self._backend == "openai":
            resp = self._openai.embeddings.create(
                input=[query],
                model=self.config.OPENAI_EMBED_MODEL,
            )
            return resp.data[0].embedding
        else:
            prefixed = f"Represent this question for searching relevant passages: {query}"
            return self._model.encode(
                [prefixed], normalize_embeddings=True
            )[0].tolist()

## CELL 7 — VECTOR STORE (unchanged from v1, well-designed)

═══════════════════════════════════════════════════════════════════════════════
CELL 7 — VECTOR STORE (unchanged from v1, well-designed)
═══════════════════════════════════════════════════════════════════════════════

In [ ]:
# @title Vector Store Manager (Qdrant)

class VectorStoreManager:
    """Manages Qdrant collection — v1 design was already production-grade."""

    def __init__(self, config: Config):
        self.config = config
        self.collection = config.COLLECTION_NAME
        if config.QDRANT_PATH:
            Path(config.QDRANT_PATH).mkdir(parents=True, exist_ok=True)
            self.client = QdrantClient(path=config.QDRANT_PATH)
        else:
            self.client = QdrantClient(":memory:")

    def ensure_collection(self, dim: int) -> None:
        existing = [c.name for c in self.client.get_collections().collections]
        if self.collection not in existing:
            self.client.create_collection(
                collection_name=self.collection,
                vectors_config=VectorParams(size=dim, distance=Distance.COSINE),
            )
            log.info("Created Qdrant collection '%s' dim=%d", self.collection, dim)

    def upsert_chunks(
        self,
        chunk_ids: List[str],
        embeddings: List[List[float]],
        payloads: List[Dict[str, Any]],
    ) -> None:
        points = [
            PointStruct(
                id=self._to_uint64(cid),
                vector=emb,
                payload={**pay, "chunk_id": cid},
            )
            for cid, emb, pay in zip(chunk_ids, embeddings, payloads)
        ]
        for i in range(0, len(points), 256):
            self.client.upsert(collection_name=self.collection, points=points[i:i + 256])

    def search(
        self,
        query_vector: List[float],
        k: int = 10,
        filter_pdf: Optional[str] = None,
    ) -> List[Dict[str, Any]]:
        qdrant_filter = None
        if filter_pdf:
            qdrant_filter = Filter(
                must=[FieldCondition(key="source_pdf", match=MatchValue(value=filter_pdf))]
            )
        results = self.client.search(
            collection_name=self.collection,
            query_vector=query_vector,
            limit=k,
            query_filter=qdrant_filter,
            with_payload=True,
        )
        return [
            {"chunk_id": r.payload.get("chunk_id", ""), "score": r.score, "payload": r.payload}
            for r in results
        ]

    @staticmethod
    def _to_uint64(uuid_str: str) -> int:
        return int(uuid_str.replace("-", "")[:16], 16)

## CELL 8 — BM25 INDEX (NEW — sparse retrieval)

═══════════════════════════════════════════════════════════════════════════════
CELL 8 — BM25 INDEX (NEW — sparse retrieval)
═══════════════════════════════════════════════════════════════════════════════

In [ ]:
# @title BM25 Sparse Index

class BM25Index:
    """
    NEW: Sparse keyword retrieval using BM25Okapi.
    Complements dense retrieval for exact keyword / number matching.
    Rebuilt from docstore on each pipeline init (fast, in-memory).
    """

    def __init__(self):
        self._corpus: List[Dict[str, Any]] = []   # chunk dicts
        self._bm25: Optional[BM25Okapi] = None
        self._tokenised: List[List[str]] = []

    def build(self, chunks: List[Dict[str, Any]]) -> None:
        """Build BM25 index from chunk dicts (must have 'content' key)."""
        self._corpus = chunks
        self._tokenised = [self._tokenise(c["content"]) for c in chunks]
        self._bm25 = BM25Okapi(self._tokenised)
        log.info("BM25 index built: %d documents", len(chunks))

    def search(self, query: str, k: int = 10) -> List[Dict[str, Any]]:
        """Returns top-k chunks with BM25 score in 'score' field."""
        if not self._bm25 or not self._corpus:
            return []
        tokens = self._tokenise(query)
        scores = self._bm25.get_scores(tokens)
        top_k = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)[:k]
        results = []
        for idx, score in top_k:
            chunk = dict(self._corpus[idx])
            chunk["score"] = float(score)
            results.append(chunk)
        return results

    @staticmethod
    def _tokenise(text: str) -> List[str]:
        """Simple whitespace + lowercase tokenisation. Adequate for BM25."""
        return re.sub(r"[^a-z0-9\s]", " ", text.lower()).split()

    @property
    def is_ready(self) -> bool:
        return self._bm25 is not None

## CELL 9 — HYBRID RETRIEVER (NEW — Reciprocal Rank Fusion)

═══════════════════════════════════════════════════════════════════════════════
CELL 9 — HYBRID RETRIEVER (NEW — Reciprocal Rank Fusion)
═══════════════════════════════════════════════════════════════════════════════

In [ ]:
# @title Hybrid Retriever (Dense + BM25 via RRF)

class HybridRetriever:
    """
    NEW: Fuses dense (Qdrant) and sparse (BM25) results using
    Reciprocal Rank Fusion (RRF) — the standard hybrid retrieval algorithm.

    RRF score = Σ 1 / (k + rank_i)   where k=60 is the standard constant.

    Why RRF over linear combination?
    - Score-scale independent (no normalisation needed)
    - Empirically outperforms linear fusion across benchmarks
    - Handles BM25 scores (unbounded) + cosine similarity (0-1) naturally
    """

    RRF_K = 60   # standard constant from original RRF paper

    def __init__(
        self,
        vector_store: VectorStoreManager,
        bm25_index: BM25Index,
        embedder: EmbeddingEngine,
        config: Config,
    ):
        self.vs = vector_store
        self.bm25 = bm25_index
        self.embedder = embedder
        self.config = config

    def retrieve(
        self,
        query: str,
        k: int = 10,
        filter_pdf: Optional[str] = None,
        use_hyde: bool = True,
    ) -> List[Dict[str, Any]]:
        """
        Hybrid retrieve with optional HyDE.
        Returns top-k chunks fused from dense + sparse results.
        """
        # ── Optional HyDE: embed a hypothetical answer instead of the query ──
        if use_hyde:
            query_vec = self._hyde_embed(query)
        else:
            query_vec = self.embedder.embed_query(query)

        # ── Dense retrieval ──────────────────────────────────────────────────
        dense_results = self.vs.search(query_vec, k=k * 2, filter_pdf=filter_pdf)
        dense_by_id: Dict[str, Dict] = {r["chunk_id"]: r for r in dense_results}

        # ── Sparse retrieval ─────────────────────────────────────────────────
        sparse_results = self.bm25.search(query, k=k * 2)
        sparse_by_id: Dict[str, Dict] = {r["id"]: r for r in sparse_results}

        # ── RRF fusion ───────────────────────────────────────────────────────
        rrf_scores: Dict[str, float] = defaultdict(float)

        for rank, result in enumerate(dense_results):
            cid = result["chunk_id"]
            rrf_scores[cid] += 1.0 / (self.RRF_K + rank + 1)

        for rank, result in enumerate(sparse_results):
            cid = result["id"]
            rrf_scores[cid] += 1.0 / (self.RRF_K + rank + 1)

        # ── Sort by RRF score and return top-k ───────────────────────────────
        sorted_ids = sorted(rrf_scores, key=rrf_scores.__getitem__, reverse=True)[:k]

        fused = []
        for cid in sorted_ids:
            if cid in dense_by_id:
                entry = dict(dense_by_id[cid])
                entry["rrf_score"] = rrf_scores[cid]
                entry["retrieval_method"] = "dense+sparse" if cid in sparse_by_id else "dense"
            else:
                # sparse-only result — build compatible dict
                sr = sparse_by_id[cid]
                entry = {
                    "chunk_id": cid,
                    "score": sr["score"],
                    "payload": {
                        "source_pdf": sr.get("source_pdf", ""),
                        "chunk_type": sr.get("chunk_type", "text"),
                        "page_num": sr.get("page_num"),
                        "chunk_id": cid,
                    },
                    "rrf_score": rrf_scores[cid],
                    "retrieval_method": "sparse",
                }
            fused.append(entry)

        return fused

    def _hyde_embed(self, query: str) -> List[float]:
        """
        NEW: Hypothetical Document Embeddings (HyDE).
        Instead of embedding the query, we ask Claude to write a short
        hypothetical answer, then embed THAT. The hypothesis lives in the
        same embedding space as real documents → better recall.

        Much cheaper than full query expansion (1 short generation, no extra retrieval).
        """
        try:
            client = anthropic.Anthropic(api_key=self.config.ANTHROPIC_API_KEY)
            resp = client.messages.create(
                model=self.config.CLAUDE_SUMMARY_MODEL,
                max_tokens=200,
                messages=[{
                    "role": "user",
                    "content": (
                        f"Write a 2-3 sentence passage that would directly answer this question "
                        f"from a corporate sustainability report. Be specific.\n\nQuestion: {query}"
                    ),
                }],
            )
            hypothesis = resp.content[0].text.strip()
            log.debug("HyDE hypothesis: %s", hypothesis[:100])
            return self.embedder.embed_query(hypothesis)
        except Exception as e:
            log.warning("HyDE failed, falling back to query embed: %s", e)
            return self.embedder.embed_query(query)

## CELL 10 — RERANKER (unchanged from v1 — already mature)

═══════════════════════════════════════════════════════════════════════════════
CELL 10 — RERANKER (unchanged from v1 — already mature)
═══════════════════════════════════════════════════════════════════════════════

In [ ]:
# @title Reranker

class Reranker:
    """Cohere / CrossEncoder reranker — v1 design retained."""

    def __init__(self, config: Config):
        self.config = config
        if _COHERE_AVAILABLE and config.COHERE_API_KEY:
            self._backend = "cohere"
            self._cohere = _cohere_lib.Client(config.COHERE_API_KEY)
            log.info("Reranker: Cohere %s", config.COHERE_RERANK_MODEL)
        elif _CROSS_ENCODER_AVAILABLE:
            self._backend = "cross_encoder"
            self._ce = CrossEncoder(config.LOCAL_RERANK_MODEL, max_length=512)
            log.info("Reranker: local CrossEncoder")
        else:
            self._backend = "none"
            log.warning("No reranker available — using ANN order")

    def rerank(
        self,
        query: str,
        candidates: List[Dict[str, Any]],
        top_n: int = 5,
    ) -> List[Dict[str, Any]]:
        if not candidates:
            return []
        if self._backend == "none" or len(candidates) <= top_n:
            return candidates[:top_n]

        texts = [c.get("display_text", c.get("content", "")) for c in candidates]

        if self._backend == "cohere":
            results = self._cohere.rerank(
                query=query,
                documents=texts,
                model=self.config.COHERE_RERANK_MODEL,
                top_n=top_n,
            )
            return [candidates[r.index] for r in results.results]
        else:
            pairs = [(query, t) for t in texts]
            scores = self._ce.predict(pairs)
            indexed = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)
            return [candidates[i] for i, _ in indexed[:top_n]]

## CELL 11 — TOKEN BUDGETER (NEW)

═══════════════════════════════════════════════════════════════════════════════
CELL 11 — TOKEN BUDGETER (NEW)
═══════════════════════════════════════════════════════════════════════════════

In [ ]:
# @title Token Budgeter (context compression)

class TokenBudgeter:
    """
    NEW: Ensures total context tokens stay within TOKEN_BUDGET.
    Strategy:
      1. Sort chunks by retrieval score (descending).
      2. Include chunks greedily until budget exhausted.
      3. If a single chunk exceeds remaining budget, truncate it.
    This prevents context window overflow and reduces LLM cost.
    """

    def __init__(self, config: Config):
        self.budget = config.TOKEN_BUDGET
        self.compress = config.COMPRESS_CONTEXT

    def pack(self, chunks: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        """Return subset of chunks that fit within token budget."""
        if not self.compress:
            return chunks

        packed = []
        remaining = self.budget

        for chunk in chunks:
            text = chunk.get("display_text", chunk.get("content", ""))
            tok = count_tokens(text)

            if tok <= remaining:
                packed.append(chunk)
                remaining -= tok
            elif remaining > 200:
                # Truncate chunk to fit remaining budget
                words = text.split()
                # Approximate: 1 token ≈ 0.75 words
                target_words = int(remaining * 0.75)
                truncated = " ".join(words[:target_words]) + " [truncated]"
                chunk = dict(chunk)
                chunk["display_text"] = truncated
                chunk["content"] = truncated
                packed.append(chunk)
                remaining = 0
                break
            else:
                break

        log.info(
            "Token budget: %d chunks selected (%d tokens budget, ~%d used)",
            len(packed),
            self.budget,
            self.budget - remaining,
        )
        return packed

## CELL 12 — ANSWER GENERATOR WITH CITATION GROUNDING (improved)

═══════════════════════════════════════════════════════════════════════════════
CELL 12 — ANSWER GENERATOR WITH CITATION GROUNDING (improved)
═══════════════════════════════════════════════════════════════════════════════

In [ ]:
# @title Answer Generator (with enforced citation grounding)

class AnswerGenerator:
    """
    v2 improvements:
      - Stricter citation enforcement in system prompt
      - Structured citation parsing: extracts inline citations into structured list
      - Hallucination guard: checks if cited pages exist in retrieved chunks
    """

    SYSTEM_PROMPT = """You are an expert analyst specialising in corporate sustainability reports.
Answer ONLY from the provided <context> passages.

CITATION RULES (strictly enforced):
- After EVERY factual claim, insert a citation in this exact format: [Source: FILENAME p.PAGE]
- Example: "ITC reduced emissions by 30% [Source: itc-report.pdf p.42]."
- If multiple sources support a claim, list all: [Source: file.pdf p.3, file.pdf p.7]
- If the context doesn't answer the question, say: "The provided documents do not contain information about [topic]."
- NEVER invent statistics, dates, or claims not present in the context.

OUTPUT FORMAT:
- Answer in clear markdown paragraphs.
- Use bullet points only for enumerated lists of 3+ items.
- End with a ## Sources section listing each unique source used (filename + page).
- Do not repeat the question."""

    def __init__(self, config: Config):
        self.client = anthropic.Anthropic(api_key=config.ANTHROPIC_API_KEY)
        self.config = config

    def generate(
        self,
        query: str,
        chunks: List[Dict[str, Any]],
    ) -> Dict[str, Any]:
        has_images = any(c.get("chunk_type") == "image" for c in chunks)
        model = (
            self.config.CLAUDE_MULTIMODAL_MODEL if has_images
            else self.config.CLAUDE_TEXT_MODEL
        )

        context_parts = self._build_context(chunks)
        user_message = self._build_user_message(query, context_parts)

        resp = self.client.messages.create(
            model=model,
            max_tokens=self.config.MAX_TOKENS,
            temperature=self.config.TEMPERATURE,
            system=self.SYSTEM_PROMPT,
            messages=[{"role": "user", "content": user_message}],
        )

        answer_text = resp.content[0].text

        # NEW: Parse inline citations and validate against retrieved chunks
        citations = self._parse_citations(answer_text)
        valid_pages = {
            (c.get("source_pdf", ""), c.get("page_num"))
            for c in chunks
        }
        hallucinated_citations = [
            cit for cit in citations
            if (cit["file"], cit["page"]) not in valid_pages
        ]
        if hallucinated_citations:
            log.warning(
                "⚠️  Possible citation hallucination: %s",
                hallucinated_citations,
            )

        return {
            "answer": answer_text,
            "model_used": model,
            "has_images": has_images,
            "chunk_count": len(chunks),
            "input_tokens": resp.usage.input_tokens,
            "output_tokens": resp.usage.output_tokens,
            "parsed_citations": citations,
            "hallucinated_citations": hallucinated_citations,  # NEW
        }

    def _parse_citations(self, answer: str) -> List[Dict[str, Any]]:
        """
        NEW: Extract structured citations from answer text.
        Parses [Source: filename.pdf p.N] patterns.
        """
        pattern = re.compile(r"\[Source:\s*([^\]]+?)\s+p\.(\d+)\]")
        citations = []
        for m in pattern.finditer(answer):
            citations.append({"file": m.group(1).strip(), "page": int(m.group(2))})
        return citations

    def _build_context(self, chunks: List[Dict[str, Any]]) -> List[Any]:
        parts = []
        for i, chunk in enumerate(chunks, 1):
            ctype = chunk.get("chunk_type", "text")
            page = chunk.get("page_num", "?")
            source = chunk.get("source_pdf", "document")
            section = chunk.get("section_title", "")

            header = (
                f"\n<context id='{i}' type='{ctype}' "
                f"source='{source}' page='{page}'"
                + (f" section='{section}'" if section else "")
                + ">"
            )

            if ctype == "image":
                parts.append({"type": "text", "text": header})
                parts.append({
                    "type": "image",
                    "source": {
                        "type": "base64",
                        "media_type": "image/png",
                        "data": chunk["content"],
                    },
                })
                if chunk.get("summary"):
                    parts.append({
                        "type": "text",
                        "text": f"[Image description: {chunk['summary']}]</context>",
                    })
                else:
                    parts.append({"type": "text", "text": "</context>"})
            else:
                raw = chunk.get("raw_content", chunk.get("content", ""))
                parts.append({
                    "type": "text",
                    "text": f"{header}\n{raw}\n</context>",
                })
        return parts

    def _build_user_message(self, query: str, context_parts: List[Any]) -> List[Any]:
        msg = [{"type": "text", "text": "<context>\n"}]
        msg.extend(context_parts)
        msg.append({
            "type": "text",
            "text": f"\n</context>\n\nQuestion: {query}\n\nAnswer with citations:",
        })
        return msg

## CELL 13 — EVALUATION SUITE (NEW — RAGAS-lite)

═══════════════════════════════════════════════════════════════════════════════
CELL 13 — EVALUATION SUITE (NEW — RAGAS-lite)
═══════════════════════════════════════════════════════════════════════════════

In [ ]:
# @title Evaluation Suite (Recall@K, MRR, faithfulness)

class RAGEvaluator:
    """
    NEW: Lightweight RAGAS-style evaluation framework.
    Measures:
      - Recall@K: did any retrieved chunk contain the answer?
      - MRR: mean reciprocal rank of the first relevant chunk
      - Faithfulness: does the answer contain only info from the context?
      - Hallucination rate: % of queries with hallucinated citations

    Usage:
        evaluator = RAGEvaluator(pipeline)
        results = evaluator.evaluate(test_cases)
        evaluator.report(results)
    """

    def __init__(self, pipeline: "RAGPipeline"):
        self.pipeline = pipeline
        self.client = anthropic.Anthropic(
            api_key=pipeline.config.ANTHROPIC_API_KEY
        )

    def evaluate(
        self,
        test_cases: List[Dict[str, Any]],
        k_values: List[int] = [1, 3, 5],
    ) -> Dict[str, Any]:
        """
        Args:
            test_cases: list of dicts with keys:
                - question (str)
                - expected_answer (str)         ← ground truth
                - relevant_chunks (List[str])    ← optional: known relevant chunk IDs
                - relevant_keywords (List[str])  ← keywords that must appear in answer

        Returns: metrics dict
        """
        recall_at_k: Dict[int, List[float]] = {k: [] for k in k_values}
        mrr_scores: List[float] = []
        faithfulness_scores: List[float] = []
        hallucination_count = 0
        total = len(test_cases)

        for tc in test_cases:
            question = tc["question"]
            expected = tc.get("expected_answer", "")
            rel_ids = set(tc.get("relevant_chunks", []))
            keywords = tc.get("relevant_keywords", [])

            result = self.pipeline.query(question)
            retrieved_ids = [s["chunk_id"] for s in result.get("_retrieved_ids", [])]

            # ── Recall@K ──────────────────────────────────────────────────────
            for k in k_values:
                top_k_ids = set(retrieved_ids[:k])
                hit = 1.0 if (rel_ids & top_k_ids) else 0.0
                recall_at_k[k].append(hit)

            # ── MRR ───────────────────────────────────────────────────────────
            rr = 0.0
            for rank, cid in enumerate(retrieved_ids, 1):
                if cid in rel_ids:
                    rr = 1.0 / rank
                    break
            mrr_scores.append(rr)

            # ── Faithfulness: keyword coverage ───────────────────────────────
            if keywords:
                answer_lower = result["answer"].lower()
                found = sum(1 for kw in keywords if kw.lower() in answer_lower)
                faithfulness_scores.append(found / len(keywords))
            else:
                faithfulness_scores.append(self._llm_faithfulness(
                    question, result["answer"], expected
                ))

            # ── Hallucination check ───────────────────────────────────────────
            if result.get("hallucinated_citations"):
                hallucination_count += 1

        metrics = {
            "total_queries": total,
            "mrr": round(sum(mrr_scores) / total, 4) if total else 0,
            "faithfulness": round(sum(faithfulness_scores) / len(faithfulness_scores), 4)
                            if faithfulness_scores else 0,
            "hallucination_rate": round(hallucination_count / total, 4) if total else 0,
        }
        for k in k_values:
            scores = recall_at_k[k]
            metrics[f"recall@{k}"] = round(sum(scores) / len(scores), 4) if scores else 0

        return metrics

    def _llm_faithfulness(
        self,
        question: str,
        answer: str,
        expected: str,
    ) -> float:
        """
        Use Claude Haiku to score answer faithfulness (0-1).
        Fallback when no expected keywords are available.
        """
        try:
            resp = self.client.messages.create(
                model="claude-haiku-4-5-20251001",
                max_tokens=10,
                messages=[{
                    "role": "user",
                    "content": (
                        f"Rate 0-10 how faithfully this answer covers the key facts "
                        f"from the expected answer. Respond with ONLY a number.\n\n"
                        f"Expected: {expected}\nActual: {answer}"
                    ),
                }],
            )
            score = float(resp.content[0].text.strip()) / 10.0
            return min(max(score, 0.0), 1.0)
        except Exception:
            return 0.5

    def report(self, metrics: Dict[str, Any]) -> None:
        """Pretty-print evaluation metrics."""
        print("\n" + "═" * 50)
        print("  RAG EVALUATION REPORT")
        print("═" * 50)
        for k, v in metrics.items():
            if isinstance(v, float):
                bar = "█" * int(v * 20)
                print(f"  {k:<25} {v:.4f}  {bar}")
            else:
                print(f"  {k:<25} {v}")
        print("═" * 50 + "\n")

## CELL 14 — AUDIT LOGGER (unchanged — already best-practice)

═══════════════════════════════════════════════════════════════════════════════
CELL 14 — AUDIT LOGGER (unchanged — already best-practice)
═══════════════════════════════════════════════════════════════════════════════

In [ ]:
# @title Audit Logger

class AuditLogger:
    """Append-only JSONL audit log for compliance + fine-tuning dataset."""

    def __init__(self, log_file: str):
        self.log_file = log_file
        Path(log_file).parent.mkdir(parents=True, exist_ok=True)

    def log(self, record: Dict[str, Any]) -> None:
        record["timestamp"] = datetime.datetime.utcnow().isoformat()
        with open(self.log_file, "a", encoding="utf-8") as f:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

## CELL 15 — MAIN PIPELINE ORCHESTRATOR

═══════════════════════════════════════════════════════════════════════════════
CELL 15 — MAIN PIPELINE ORCHESTRATOR
═══════════════════════════════════════════════════════════════════════════════

In [ ]:
# @title RAGPipeline (main orchestrator)

class RAGPipeline:
    """
    Production multimodal RAG pipeline — v2.

    Public API:
        pipeline = RAGPipeline()
        pipeline.add_pdf("report.pdf")
        result = pipeline.query("What is ITC's water target?")
        print(result["answer"])

    New in v2:
        - Hybrid dense+sparse retrieval (BM25 + Qdrant)
        - HyDE query embeddings
        - Async concurrent summarisation
        - Token-budgeted context packing
        - Citation grounding with hallucination detection
        - OCR fallback for scanned PDFs
        - Embedding disk cache
        - Evaluation suite (RAGEvaluator)
    """

    def __init__(self, config: Optional[Config] = None):
        self.config = config or Config()
        self.config.validate()
        log.info("Initialising RAG pipeline v2 …")

        self.docstore     = DocStore(self.config.DOCSTORE_PATH)
        self.parser       = PDFParser(self.config)
        self.summariser   = AsyncSummariser(self.config)
        self.embedder     = EmbeddingEngine(self.config)

        self.vector_store = VectorStoreManager(self.config)
        self.vector_store.ensure_collection(self.config.EMBED_DIM)

        # BM25 index — rebuilt from docstore on init
        self.bm25_index   = BM25Index()
        self._rebuild_bm25()

        self.retriever    = HybridRetriever(
            self.vector_store, self.bm25_index, self.embedder, self.config
        )
        self.reranker     = Reranker(self.config)
        self.budgeter     = TokenBudgeter(self.config)
        self.generator    = AnswerGenerator(self.config)
        self.audit        = AuditLogger(self.config.LOG_FILE)

        log.info("Pipeline v2 ready. ✓")

    # ── BM25 management ──────────────────────────────────────────────────────

    def _rebuild_bm25(self) -> None:
        """Rebuild BM25 index from all stored chunks."""
        chunks = self.docstore.get_all_for_bm25()
        if chunks:
            self.bm25_index.build(chunks)
        else:
            log.info("BM25: no existing chunks, will build after first PDF indexed.")

    # ── Public: Index a PDF ──────────────────────────────────────────────────

    def add_pdf(self, pdf_path: str, force_reindex: bool = False) -> None:
        """
        Index a PDF. Idempotent — skips if already indexed.
        v2: async summarisation, content-addressed dedup, BM25 rebuild.
        """
        pdf_name = Path(pdf_path).name

        if not force_reindex and self.docstore.pdf_exists(pdf_name):
            log.info("'%s' already indexed. Skipping. (force_reindex=True to override)", pdf_name)
            return

        log.info("=== Indexing: %s ===", pdf_name)

        # Step 1: Parse
        chunks = self.parser.parse(pdf_path)

        # Step 2: Content-addressed dedup — skip chunks with identical content
        new_chunks = []
        dedup_count = 0
        for chunk in chunks:
            if chunk["chunk_type"] == "image":
                new_chunks.append(chunk)   # images: always include (b64 compare expensive)
                continue
            existing = self.docstore.content_exists(chunk["content"])
            if existing:
                dedup_count += 1
            else:
                new_chunks.append(chunk)

        if dedup_count:
            log.info("Deduped %d chunks with identical content", dedup_count)
        chunks = new_chunks

        # Step 3: Async summarisation (concurrent)
        log.info("Summarising %d chunks concurrently …", len(chunks))
        chunks = asyncio.run(self.summariser.summarise_batch(chunks))

        # Step 4: Embed
        texts_to_embed = [
            (chunk.get("summary") or chunk["content"])
            if chunk["chunk_type"] in ("table", "image")
            else chunk["content"]
            for chunk in chunks
        ]
        log.info("Embedding %d chunks …", len(texts_to_embed))
        embeddings = self.embedder.embed(texts_to_embed)

        # Step 5: Store
        log.info("Storing chunks …")
        ids, vecs, payloads = [], [], []
        for chunk, emb in zip(chunks, embeddings):
            self.docstore.upsert(chunk)
            ids.append(chunk["id"])
            vecs.append(emb)
            payloads.append({
                "source_pdf":    chunk["source_pdf"],
                "chunk_type":    chunk["chunk_type"],
                "page_num":      chunk.get("page_num"),
                "section_title": chunk.get("section_title", ""),
                "summary":       chunk.get("summary", ""),
                "text_preview":  chunk["content"][:500]
                                 if chunk["chunk_type"] in ("text", "table")
                                 else "[image]",
            })

        self.vector_store.upsert_chunks(ids, vecs, payloads)

        # Step 6: Rebuild BM25 index with new chunks
        self._rebuild_bm25()

        log.info("✓ Indexed '%s' — %d chunks.", pdf_name, len(chunks))

    # ── Public: Query ─────────────────────────────────────────────────────────

    def query(
        self,
        question: str,
        filter_pdf: Optional[str] = None,
        use_hyde: bool = True,
    ) -> Dict[str, Any]:
        """
        End-to-end RAG query with hybrid retrieval.

        Args:
            question:    The user's question.
            filter_pdf:  Optional — restrict to a single PDF.
            use_hyde:    Use HyDE query embedding (default True, better recall).

        Returns dict with:
            answer, model_used, has_images, sources,
            parsed_citations, hallucinated_citations, latency_s, tokens
        """
        t0 = time.time()

        # Step 1: Hybrid retrieve (dense + BM25 + RRF + optional HyDE)
        retrieved = self.retriever.retrieve(
            query=question,
            k=self.config.RETRIEVAL_K,
            filter_pdf=filter_pdf,
            use_hyde=use_hyde,
        )
        log.info("Retrieved %d candidates (hybrid)", len(retrieved))

        # Step 2: Hydrate from docstore (get full content)
        hydrated: List[Dict[str, Any]] = []
        for cand in retrieved:
            stored = self.docstore.get(cand["chunk_id"])
            if stored:
                stored["retrieval_score"] = cand.get("rrf_score", cand.get("score", 0))
                stored["retrieval_method"] = cand.get("retrieval_method", "dense")
                stored["display_text"] = (
                    stored.get("summary") or stored["content"]
                    if stored["chunk_type"] in ("table", "image")
                    else stored["content"]
                )
                stored["raw_content"] = stored["content"]
                hydrated.append(stored)

        # Step 3: Rerank
        reranked = self.reranker.rerank(
            query=question,
            candidates=hydrated,
            top_n=self.config.RERANK_TOP_N,
        )
        log.info("Reranked to top %d", len(reranked))

        # Step 4: Token budget — compress context to fit
        budgeted = self.budgeter.pack(reranked)

        # Step 5: Generate answer
        result = self.generator.generate(question, budgeted)

        # Step 6: Build source list
        sources = [
            {
                "chunk_id":   c["id"],
                "source_pdf": c["source_pdf"],
                "page_num":   c["page_num"],
                "chunk_type": c["chunk_type"],
                "section":    c.get("section_title", ""),
                "method":     c.get("retrieval_method", "dense"),
                "score":      round(c["retrieval_score"], 4),
            }
            for c in budgeted
        ]

        t1 = time.time()
        result.update({
            "question":          question,
            "sources":           sources,
            "latency_s":         round(t1 - t0, 2),
            "_retrieved_ids":    retrieved,   # for evaluation
        })

        # Step 7: Audit log
        self.audit.log({
            "question":               question,
            "model_used":             result["model_used"],
            "has_images":             result["has_images"],
            "chunk_count":            result["chunk_count"],
            "latency_s":              result["latency_s"],
            "input_tokens":           result["input_tokens"],
            "output_tokens":          result["output_tokens"],
            "retrieval_method":       "hybrid+HyDE" if use_hyde else "hybrid",
            "hallucinated_citations": result.get("hallucinated_citations", []),
            "sources":                sources,
        })

        return result

## CELL 16 — DISPLAY HELPERS

═══════════════════════════════════════════════════════════════════════════════
CELL 16 — DISPLAY HELPERS
═══════════════════════════════════════════════════════════════════════════════

In [ ]:
# @title Display helpers

def display_result(result: Dict[str, Any]) -> None:
    """Pretty-print a query result."""
    if _RICH and console:
        console.print()
        title = (
            f"[bold green]Answer[/bold green] — {result['model_used']} "
            f"({'📸 multimodal' if result['has_images'] else '📝 text'}) "
            f"| hybrid+HyDE | {result['latency_s']}s"
        )
        console.print(Panel(Markdown(result["answer"]), title=title, border_style="green"))

        if result.get("sources"):
            lines = []
            for s in result["sources"]:
                method_badge = "🔵" if "dense" in s["method"] else "🟡"
                lines.append(
                    f"  {method_badge} {s['source_pdf']} p.{s['page_num']} "
                    f"[{s['chunk_type']}] {s.get('section','')[:40]} (rrf={s['score']})"
                )
            console.print(Panel("\n".join(lines), title="[bold blue]Sources[/bold blue]", border_style="blue"))

        if result.get("hallucinated_citations"):
            console.print(
                f"[bold red]⚠️  Possible hallucinated citations: "
                f"{result['hallucinated_citations']}[/bold red]"
            )
        console.print(
            f"[dim]Tokens: {result['input_tokens']} in / {result['output_tokens']} out[/dim]"
        )
    else:
        print("\n" + "=" * 70)
        print("ANSWER:")
        print(result["answer"])
        print("\nSOURCES:")
        for s in result.get("sources", []):
            print(f"  [{s['method']}] {s['source_pdf']} p.{s['page_num']} [{s['chunk_type']}]")
        if result.get("hallucinated_citations"):
            print(f"\n⚠️  Hallucinated citations: {result['hallucinated_citations']}")
        print(f"\nLatency: {result['latency_s']}s | Model: {result['model_used']}")
        print("=" * 70)

## CELL 17 — INTERACTIVE REPL

═══════════════════════════════════════════════════════════════════════════════
CELL 17 — INTERACTIVE REPL
═══════════════════════════════════════════════════════════════════════════════

In [ ]:
# @title Interactive REPL

def run_repl(pipeline: RAGPipeline) -> None:
    """Interactive REPL. Type 'exit', 'eval', or 'help'."""
    print("\n=== RAG Pipeline v2 Ready ===")
    print("Commands: exit · help · eval")
    print("Retrieval: Hybrid BM25+dense · HyDE · Token budgeted\n")

    while True:
        try:
            question = input("Query > ").strip()
        except (KeyboardInterrupt, EOFError):
            print("\nGoodbye.")
            break

        if not question:
            continue
        if question.lower() in ("exit", "quit", "q"):
            break
        if question.lower() == "help":
            print("Type any question about the indexed documents.")
            print("Prefixes: 'hyde off: <q>' to disable HyDE for a query")
            continue
        if question.lower() == "eval":
            print("No test cases defined. Create test_cases list and run RAGEvaluator.")
            continue

        use_hyde = True
        if question.lower().startswith("hyde off:"):
            use_hyde = False
            question = question[9:].strip()

        try:
            result = pipeline.query(question, use_hyde=use_hyde)
            display_result(result)
        except Exception as e:
            log.error("Query failed: %s", e, exc_info=True)
            print(f"Error: {e}")

## CELL 18 — COLAB QUICKSTART

═══════════════════════════════════════════════════════════════════════════════
CELL 18 — COLAB QUICKSTART
Copy this cell into your Colab notebook to get started immediately.
═══════════════════════════════════════════════════════════════════════════════

In [ ]:
# @title Quickstart — run this to index and query

def colab_quickstart(pdf_path: str, question: str):
    """
    Colab one-shot demo.
    Usage:
        colab_quickstart("itc-sustainability-report-2025.pdf",
                         "What is ITC's 2030 renewable energy target?")
    """
    config = Config()
    # Set your key before running:
    # config.ANTHROPIC_API_KEY = "sk-ant-..."

    pipeline = RAGPipeline(config)
    pipeline.add_pdf(pdf_path)
    result = pipeline.query(question)
    display_result(result)
    return result

## CELL 19 — CLI ENTRYPOINT (for non-Colab usage)

═══════════════════════════════════════════════════════════════════════════════
CELL 19 — CLI ENTRYPOINT (for non-Colab usage)
═══════════════════════════════════════════════════════════════════════════════

In [ ]:
# @title CLI entrypoint

import argparse

def main():
    parser = argparse.ArgumentParser(
        description="Production Multimodal RAG Pipeline v2",
        formatter_class=argparse.RawDescriptionHelpFormatter,
        epilog=textwrap.dedent("""
        Examples:
          python rag_pipeline_v2.py --pdf report.pdf
          python rag_pipeline_v2.py --pdf report.pdf --query "Water target?"
          python rag_pipeline_v2.py --query "Scope 1 emissions" --no-hyde
          python rag_pipeline_v2.py --pdf a.pdf --pdf b.pdf
        """),
    )
    parser.add_argument("--pdf", action="append", default=[], metavar="PATH")
    parser.add_argument("--query", type=str, default=None)
    parser.add_argument("--force-reindex", action="store_true")
    parser.add_argument("--no-hyde", action="store_true", help="Disable HyDE query embedding")
    args = parser.parse_args()

    config = Config()
    pipeline = RAGPipeline(config)

    for pdf_path in args.pdf:
        if not Path(pdf_path).exists():
            log.error("PDF not found: %s", pdf_path)
            sys.exit(1)
        pipeline.add_pdf(pdf_path, force_reindex=args.force_reindex)

    if args.query:
        result = pipeline.query(args.query, use_hyde=not args.no_hyde)
        display_result(result)
    else:
        run_repl(pipeline)


if __name__ == "__main__":
    main()